In [2]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Get Sentences using Ingestion tool

In [39]:
from components import *

sentences = ingest_using_url("https://jamesclear.com/saying-no")
print(sentences)

['The Ultimate Productivity Hack is Saying No - James Clear The Ultimate Productivity Hack is Saying No written by James Clear Decision Making Focus Life Lessons The ultimate productivity hack is saying no.', 'Not doing something will always be faster than doing it.', 'This statement reminds me of the old computer programming saying, “Remember that there is no code faster than no code.” 1 The same philosophy applies in other areas of life.', 'For example, there is no meeting that goes faster than not having a meeting at all.', "This is not to say you should never attend another meeting, but the truth is that we say yes to many things we don't actually want to do.", "There are many meetings held that don't need to be held.", 'There is a lot of code written that could be deleted.', "How often do people ask you to do something and you just reply, “Sure thing.” Three days later, you're overwhelmed by how much is on your to-do list.", 'We become frustrated by our obligations even though we 

### 1. TextRank (Graph-based)

In [40]:
# BACKGROUND: What is TF-IDF?

def build_tfidf_vectors(sentences: list[str]) -> tuple[np.ndarray, object]:
    """
    Convert sentences to TF-IDF vectors.
 
    Returns
    -------
    matrix     : shape (n_sentences, vocab_size)  — one row per sentence
    vectorizer : fitted TfidfVectorizer (kept for later use if needed)
    """
    vectorizer = TfidfVectorizer(
        stop_words='english',   # ignore 'the', 'is', 'and', etc.
        min_df=1,               # include words that appear at least once
        max_df=0.95,            # ignore words in >95% of sentences (too common)
    )
    matrix = vectorizer.fit_transform(sentences).toarray()
    return matrix.astype('float32'), vectorizer

In [47]:
# 2a. TEXTRANK

def build_similarity_matrix(sentences: list[str], method) -> np.ndarray:
    """
    Build an N×N matrix where entry [i,j] = cosine similarity between
    sentence i and sentence j.
 
    This is the adjacency matrix / edge weights of the TextRank graph.
    Self-similarity (diagonal) is set to 0 so a sentence doesn't vote for itself.
 
    Example for 3 sentences:
         S1    S2    S3
    S1 [ 0.0   0.8   0.2 ]
    S2 [ 0.8   0.0   0.3 ]
    S3 [ 0.2   0.3   0.0 ]
    """
    if len(sentences) < 2:
        return np.ones((len(sentences), len(sentences)))
 
    matrix, _ = method(sentences)
    sim = cosine_similarity(matrix)
    np.fill_diagonal(sim, 0.0)   # remove self-loops
    return sim
 
 
def textrank_scores(sentences: list[str],
                    embedd_method,
                    damping: float = 0.85,
                    max_iterations: int = 100,
                    tolerance: float = 1e-4) -> np.ndarray:
    """
    Run the TextRank power iteration algorithm.
 
    Parameters
    ----------
    damping        : probability of following a graph edge (vs. random jump)
                     higher = more influence from graph structure
    max_iterations : stop after this many rounds even if not converged
    tolerance      : stop early if scores change by less than this
 
    Returns
    -------
    scores : np.ndarray of shape (n_sentences,), normalised to [0, 1]
    """
    n = len(sentences)
    if n == 0:
        return np.array([])
    if n == 1:
        return np.array([1.0])
 
    # Step 1: build the similarity graph
    sim_matrix = build_similarity_matrix(sentences, embedd_method)
 
    # Step 2: row-normalise → transition probabilities
    # Each row sums to 1, so sim_matrix[i] = probability of moving FROM i
    row_sums = sim_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1              # avoid divide-by-zero for isolated nodes
    transition = sim_matrix / row_sums
 
    # Step 3: initialise all scores equally
    scores = np.ones(n) / n
 
    # Step 4: power iteration
    for iteration in range(max_iterations):
        # Each sentence's new score = random jump + weighted sum of neighbours' scores
        # transition.T @ scores  →  how much "vote" flows INTO each sentence
        new_scores = (1 - damping) / n + damping * (transition.T @ scores)
 
        # Check convergence — stop if scores barely changed
        delta = np.linalg.norm(new_scores - scores)
        scores = new_scores
 
        if delta < tolerance:
            print(f"    TextRank converged at iteration {iteration + 1}")
            break
 
    # Step 5: normalise final scores to [0, 1]
    s_min, s_max = scores.min(), scores.max()
    if s_max > s_min:
        scores = (scores - s_min) / (s_max - s_min)
 
    return scores

In [49]:
tfidf_sentences = build_tfidf_vectors(sentences)
print(tfidf_sentences)

tfidf = textrank_scores(sentences, embedd_method=build_tfidf_vectors)
print(tfidf)
print(f'Number of scores: {len(tfidf)}')

(array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.39481053],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.17619784, ..., 0.        , 0.        ,
        0.        ]], shape=(94, 442), dtype=float32), TfidfVectorizer(max_df=0.95, stop_words='english'))
    TextRank converged at iteration 10
[0.40760839 0.21876531 0.30221035 0.17833154 0.81808277 0.19083816
 0.06284901 0.18969399 0.19969467 0.30219856 0.04511621 0.81169843
 0.48538392 0.22703321 0.57258166 0.19118333 0.1847255  0.23809998
 0.02787638 0.19722774 0.16767802 0.29192777 0.39211184 0.32417216
 0.2071907  0.75673206 1.     

### 2. Centroid Similarity

In [50]:
# 2b. CENTROID SIMILARITY

def centroid_scores(sentences: list[str]) -> np.ndarray:
    """
    Score each sentence by cosine similarity to the document centroid.
 
    A high score means: "this sentence captures the overall topic well."
    A low score means: "this sentence is off-topic or tangential."
 
    Returns
    -------
    scores : np.ndarray of shape (n_sentences,), normalised to [0, 1]
    """
    if not sentences:
        return np.array([])
    if len(sentences) == 1:
        return np.array([1.0])
 
    # Step 1: vectorise
    matrix, _ = build_tfidf_vectors(sentences)
 
    # Step 2: compute centroid — mean of all sentence vectors
    centroid = matrix.mean(axis=0, keepdims=True)   # shape: (1, vocab_size)
 
    # Step 3: cosine similarity between each sentence and the centroid
    # cosine_similarity returns shape (n_sentences, 1) — flatten to 1D
    scores = cosine_similarity(matrix, centroid).flatten()
 
    # Step 4: normalise to [0, 1]
    s_min, s_max = scores.min(), scores.max()
    if s_max > s_min:
        scores = (scores - s_min) / (s_max - s_min)
 
    return scores

In [51]:
centroid = centroid_scores(sentences)
print(centroid)
print(f'Number of scores: {len(centroid)}')

[0.34266296 0.16130984 0.2294782  0.1493493  0.78721    0.1823673
 0.04826523 0.16795005 0.19435804 0.23777656 0.04040323 0.798114
 0.4774249  0.21909489 0.5121141  0.17714998 0.13559324 0.2114069
 0.02077006 0.20963407 0.15843125 0.26671052 0.3339536  0.3122815
 0.1723222  0.7645239  1.         0.7899403  0.62315494 0.84448016
 0.31375232 0.34324035 0.53595895 0.07953612 0.22820316 0.14684042
 0.27932715 0.17996895 0.43437836 0.4790214  0.5913425  0.47140828
 0.21707273 0.5311785  0.07465812 0.32387277 0.67237705 0.48696557
 0.1193881  0.15152414 0.26478195 0.31054008 0.21742715 0.41211182
 0.6766283  0.11619984 0.33474845 0.6398895  0.7257023  0.74309427
 0.66278946 0.13805932 0.22074543 0.3821763  0.09487666 0.14004359
 0.11703077 0.42213553 0.3984312  0.35776383 0.06751664 0.2607901
 0.29924056 0.06131178 0.0744226  0.26898757 0.24131583 0.23258151
 0.2913318  0.37878567 0.05610848 0.21281308 0.20973074 0.506104
 0.4169358  0.38065484 0.09266576 0.1336697  0.09376426 0.03589895
 0.

### 3. Combine

In [55]:
# 2c. COMBINE BOTH SIGNALS

def multi_signal_scores(sentences: list[str],
                        embed_method,
                        textrank_weight: float = 0.5,
                        centroid_weight: float = 0.5) -> dict:
    """
    Run both scoring methods and combine them into one score per sentence.
 
    Parameters
    ----------
    textrank_weight : how much to weight the TextRank score  (default 0.5)
    centroid_weight : how much to weight the centroid score  (default 0.5)
                      weights don't have to sum to 1, but it's cleaner if they do
 
    Returns
    -------
    dict with keys:
        textrank  : raw TextRank scores
        centroid  : raw centroid scores
        combined  : weighted combination of both
    """
    print("  Running TextRank scoring...")
    tr_scores  = textrank_scores(sentences, embed_method)
 
    print("  Running centroid similarity scoring...")
    cen_scores = centroid_scores(sentences)
 
    combined = textrank_weight * tr_scores + centroid_weight * cen_scores
 
    # Re-normalise combined score to [0, 1]
    c_min, c_max = combined.min(), combined.max()
    if c_max > c_min:
        combined = (combined - c_min) / (c_max - c_min)
 
    return {
        "textrank":  tr_scores,
        "centroid":  cen_scores,
        "combined":  combined,
    }

In [56]:
combined = multi_signal_scores(sentences, embed_method=build_tfidf_vectors)
print(combined)
print(f'Number of scores: {len(combined['combined'])}')


  Running TextRank scoring...
    TextRank converged at iteration 10
  Running centroid similarity scoring...
{'textrank': array([0.40760839, 0.21876531, 0.30221035, 0.17833154, 0.81808277,
       0.19083816, 0.06284901, 0.18969399, 0.19969467, 0.30219856,
       0.04511621, 0.81169843, 0.48538392, 0.22703321, 0.57258166,
       0.19118333, 0.1847255 , 0.23809998, 0.02787638, 0.19722774,
       0.16767802, 0.29192777, 0.39211184, 0.32417216, 0.2071907 ,
       0.75673206, 1.        , 0.82489508, 0.64906986, 0.87767915,
       0.31883634, 0.35003436, 0.54984645, 0.09527988, 0.24534421,
       0.18045802, 0.31188059, 0.20963703, 0.54653063, 0.4964235 ,
       0.58661437, 0.52629075, 0.2219193 , 0.63391584, 0.10239044,
       0.33036873, 0.69303169, 0.48592753, 0.13438222, 0.19525675,
       0.28297609, 0.32960082, 0.22448712, 0.43495892, 0.69322851,
       0.14806061, 0.34162969, 0.64021882, 0.74954807, 0.76196101,
       0.6413679 , 0.17893152, 0.2876743 , 0.44219129, 0.12281436,
      

### SBERT Upgrade (optional)

In [58]:
# To use SBERT, install: pip install sentence-transformers
# Then replace build_tfidf_vectors() with this:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')  # small, fast model

def build_sbert_vectors(sentences):
    return (model.encode(sentences, convert_to_numpy=True), 0)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13597.30it/s]


In [60]:
combined = multi_signal_scores(sentences, embed_method=build_sbert_vectors)
print(combined)
print(f'Number of scores: {len(combined['combined'])}')

  Running TextRank scoring...
    TextRank converged at iteration 4
  Running centroid similarity scoring...
{'textrank': array([0.61696279, 0.73898049, 0.74861518, 0.68081662, 0.75885595,
       0.52954574, 0.23609917, 0.65849557, 0.67317963, 0.60026477,
       0.64742023, 0.85287785, 0.67096777, 0.65234115, 0.61688448,
       0.42824259, 0.58483711, 0.60838989, 0.50430082, 0.5611102 ,
       0.80948948, 0.65296643, 0.74135029, 0.66626882, 0.40319001,
       0.85563732, 0.86974472, 0.91250557, 0.94048152, 1.        ,
       0.50599421, 0.71710875, 0.36861888, 0.4497118 , 0.83567322,
       0.7175881 , 0.76943177, 0.64505942, 0.9027726 , 0.76933368,
       0.8336261 , 0.76884678, 0.41524416, 0.70546402, 0.4412802 ,
       0.8777571 , 0.77520865, 0.84735919, 0.52256876, 0.35777317,
       0.71994548, 0.68125627, 0.57039014, 0.88274868, 0.78682017,
       0.49167869, 0.73810837, 0.69530775, 0.78278991, 0.92301036,
       0.79472925, 0.52276054, 0.59122908, 0.73796464, 0.57471833,
       